# AutoGaze — 토큰 효율성 & 처리량 분석

이 노트북은 AutoGaze의 핵심 가치 중 하나인 **저장 용량 절감 및 처리량 향상**을 수치로 측정합니다.

## 분석 항목

| # | 항목 | 설명 |
|---|---|---|
| 1 | **저장 용량 비교** | MP4 → 원시 픽셀 → 전체 패치 → Gaze 패치 → SigLIP 피처 |
| 2 | **Throughput 벤치마크** | AutoGaze 없이 vs. 있을 때 실제 처리 시간 |
| 3 | **Agent 반복 접근 시나리오** | 같은 비디오를 N번 반복 처리할 때 캐싱 전략 비교 |
| 4 | **데이터셋 스케일 추정** | 대량 비디오 데이터셋의 총 저장 용량 비교 |

## 핵심 질문

- AutoGaze로 선택된 토큰만 저장하면 **얼마나 공간을 절약**할 수 있나?
- 같은 비디오를 반복 처리해야 하는 Agent 워크플로에서 **gaze 캐싱이 언제 이득**인가?
- 수백만 비디오 데이터셋에서 **토큰 캐시 전략별 총 비용**은?


---

## 0. 환경 설정

In [ ]:
import os
import time
import math
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

# 한글 폰트 (macOS)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

REPO_ROOT = Path("..")
VIDEO_PATH = REPO_ROOT / "assets" / "example_input.mp4"
MODEL_PATH = REPO_ROOT / "weights" / "AutoGaze"
print(f"비디오 경로: {VIDEO_PATH.resolve()}")
print(f"비디오 존재: {VIDEO_PATH.exists()}")

---

## 1. 비디오 로드 및 AutoGaze 실행

In [ ]:
import av
from autogaze.datasets.video_utils import read_video_pyav, transform_video_for_pytorch
from autogaze.models.autogaze import AutoGazeImageProcessor, AutoGaze

# ── 비디오 메타데이터 ──────────────────────────────────
container = av.open(str(VIDEO_PATH))
stream = container.streams.video[0]
total_frames = stream.frames or sum(1 for _ in container.decode(video=0))
fps = float(stream.average_rate)
width = stream.width
height = stream.height
duration_sec = total_frames / fps if fps else 0
container.close()

mp4_bytes = VIDEO_PATH.stat().st_size

print(f"해상도      : {width} × {height}")
print(f"총 프레임   : {total_frames} frames @ {fps:.1f} fps")
print(f"길이        : {duration_sec:.2f} 초")
print(f"MP4 파일 크기: {mp4_bytes / 1024:.1f} KB")

In [ ]:
# ── 모델 로드 ──────────────────────────────────────────
transform = AutoGazeImageProcessor.from_pretrained(str(MODEL_PATH))
model = AutoGaze.from_pretrained(str(MODEL_PATH))
model.eval()
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"디바이스    : {device}")

NUM_SAMPLE_FRAMES = model.config.max_num_frames   # 기본 16
NUM_TOKENS_PER_FRAME = 265                         # 32+64+112+224 멀티스케일 합산
PATCH_PIX = 16 * 16 * 3                            # 단일 패치 픽셀 수 (16×16 RGB)
SIGLIP_DIM = 768                                   # SigLIP-base 피처 차원

print(f"샘플 프레임 : {NUM_SAMPLE_FRAMES}")
print(f"프레임당 토큰: {NUM_TOKENS_PER_FRAME} (멀티스케일 합산)")

In [ ]:
# ── 비디오 로드 → 전처리 ──────────────────────────────
container = av.open(str(VIDEO_PATH))
indices = list(range(min(NUM_SAMPLE_FRAMES, total_frames)))
raw_video = read_video_pyav(container, indices)
container.close()

video_input = transform_video_for_pytorch(raw_video, transform)  # (T, C, H, W)
T_actual = video_input.shape[0]
_, C, H, W = video_input.shape
video_batch = video_input[None].to(device)  # (1, T, C, H, W)

print(f"로드된 프레임: {T_actual}, 해상도: {C}×{H}×{W}")

In [ ]:
# ── 여러 gazing_ratio에서 AutoGaze 실행 (실제 선택 토큰 수 측정) ──
GAZING_RATIOS = [0.10, 0.25, 0.50, 0.75]
gaze_results = {}

for ratio in GAZING_RATIOS:
    # warm-up
    with torch.inference_mode():
        _ = model(
            {"video": video_batch},
            gazing_ratio=ratio,
            task_loss_requirement=None,
        )

    # 실제 측정
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model(
            {"video": video_batch},
            gazing_ratio=ratio,
            task_loss_requirement=None,
        )
    t1 = time.perf_counter()

    n_total = T_actual * NUM_TOKENS_PER_FRAME
    n_gaze  = int((~out['if_padded_gazing'][0]).sum())
    actual_ratio = n_gaze / n_total

    gaze_results[ratio] = {
        "n_gaze": n_gaze,
        "n_total": n_total,
        "actual_ratio": actual_ratio,
        "latency_ms": (t1 - t0) * 1000,
    }
    print(f"gazing_ratio={ratio:.2f}:  "
          f"{n_gaze:4d} / {n_total} 토큰 선택 "
          f"(실제 {actual_ratio*100:.1f}%)  "
          f"→ {t1-t0:.3f}s")

---

## 2. 저장 용량 비교 (바이트 단위)

같은 비디오 클립을 **어떤 형태로 저장하느냐**에 따라 용량이 크게 다릅니다.

In [ ]:
def fmt_bytes(b: int) -> str:
    """사람이 읽기 쉬운 크기 문자열"""
    if b < 1024:
        return f"{b} B"
    elif b < 1024**2:
        return f"{b/1024:.1f} KB"
    elif b < 1024**3:
        return f"{b/1024**2:.2f} MB"
    else:
        return f"{b/1024**3:.2f} GB"


# ── 표현 방식별 크기 계산 ─────────────────────────────
# 1. MP4 (압축)
b_mp4 = mp4_bytes

# 2. 원시 픽셀 — uint8 (디코딩된 프레임, 8비트)
b_raw_uint8 = T_actual * C * H * W * 1   # 1 byte/channel

# 3. 원시 픽셀 — float32 (모델 입력 텐서)
b_raw_float32 = T_actual * C * H * W * 4  # 4 bytes/channel

# 4. 멀티스케일 전체 패치 — float16 (265 토큰 × 768 픽셀값)
#    각 패치는 패치 크기 16×16×3=768 값, 모든 스케일 공통
b_full_patch_fp16 = T_actual * NUM_TOKENS_PER_FRAME * PATCH_PIX * 2  # float16

# 5. Gaze 선택 패치 — float16 (각 ratio별)
b_gaze_patch_fp16 = {
    ratio: res["n_gaze"] * PATCH_PIX * 2
    for ratio, res in gaze_results.items()
}

# 6. SigLIP 피처 — float16 (인코딩 후)
b_full_siglip_fp16 = T_actual * NUM_TOKENS_PER_FRAME * SIGLIP_DIM * 2
b_gaze_siglip_fp16 = {
    ratio: res["n_gaze"] * SIGLIP_DIM * 2
    for ratio, res in gaze_results.items()
}

print("=" * 55)
print(f"{'표현 방식':<35} {'크기':>10}  {'MP4 대비':>8}")
print("=" * 55)
print(f"{'MP4 (압축 원본)':<35} {fmt_bytes(b_mp4):>10}  {'1.00×':>8}")
print(f"{'원시 픽셀 uint8':<35} {fmt_bytes(b_raw_uint8):>10}  {b_raw_uint8/b_mp4:>7.1f}×")
print(f"{'원시 픽셀 float32':<35} {fmt_bytes(b_raw_float32):>10}  {b_raw_float32/b_mp4:>7.1f}×")
print(f"{'전체 패치 float16 (265 tok)':<35} {fmt_bytes(b_full_patch_fp16):>10}  {b_full_patch_fp16/b_mp4:>7.1f}×")
print("-" * 55)
for ratio in GAZING_RATIOS:
    b = b_gaze_patch_fp16[ratio]
    label = f"  Gaze 패치 fp16  ratio={ratio:.2f}"
    print(f"{label:<35} {fmt_bytes(b):>10}  {b/b_mp4:>7.2f}×  ({b/b_full_patch_fp16*100:.0f}% of full)")
print("=" * 55)
print(f"{'전체 SigLIP 피처 fp16':<35} {fmt_bytes(b_full_siglip_fp16):>10}  {b_full_siglip_fp16/b_mp4:>7.1f}×")
for ratio in GAZING_RATIOS:
    b = b_gaze_siglip_fp16[ratio]
    label = f"  Gaze SigLIP fp16 ratio={ratio:.2f}"
    print(f"{label:<35} {fmt_bytes(b):>10}  {b/b_mp4:>7.2f}×  ({b/b_full_siglip_fp16*100:.0f}% of full)")
print("=" * 55)

In [ ]:
# ── 저장 용량 비교 바 차트 ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 왼쪽: 픽셀/패치 수준 비교 ──
labels_left = [
    "MP4\n(압축)",
    "원시픽셀\nuint8",
    "원시픽셀\nfloat32",
    "전체 패치\nfp16",
    "Gaze 패치\n0.25 fp16",
    "Gaze 패치\n0.10 fp16",
]
vals_left = [
    b_mp4,
    b_raw_uint8,
    b_raw_float32,
    b_full_patch_fp16,
    b_gaze_patch_fp16[0.25],
    b_gaze_patch_fp16[0.10],
]
colors_left = ["#4C72B0", "#DD8452", "#C44E52", "#8172B2", "#55A868", "#2E8B57"]
ax = axes[0]
bars = ax.bar(labels_left, [v / 1024**2 for v in vals_left], color=colors_left, edgecolor="white", width=0.6)
for bar, v in zip(bars, vals_left):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            fmt_bytes(v), ha="center", va="bottom", fontsize=8)
ax.set_ylabel("MB")
ax.set_title("픽셀 / 패치 수준 저장 용량 비교")
ax.set_ylim(0, max(vals_left) / 1024**2 * 1.25)

# ── 오른쪽: SigLIP 피처 수준 비교 ──
labels_right = ["전체\nSigLIP fp16"] + [f"Gaze SigLIP\n{r:.2f}" for r in GAZING_RATIOS]
vals_right  = [b_full_siglip_fp16] + [b_gaze_siglip_fp16[r] for r in GAZING_RATIOS]
colors_right = ["#8172B2"] + ["#55A868", "#4DAF4A", "#2CA02C", "#006400"]
ax2 = axes[1]
bars2 = ax2.bar(labels_right, [v / 1024**2 for v in vals_right], color=colors_right, edgecolor="white", width=0.6)
for bar, v in zip(bars2, vals_right):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             fmt_bytes(v), ha="center", va="bottom", fontsize=8)
ax2.set_ylabel("MB")
ax2.set_title("SigLIP 피처 캐시 저장 용량 비교")
ax2.set_ylim(0, vals_right[0] / 1024**2 * 1.3)

# 절감률 주석
for bar, v in zip(bars2[1:], vals_right[1:]):
    pct = v / vals_right[0] * 100
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() / 2,
             f"{pct:.0f}%", ha="center", va="center",
             color="white", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("storage_comparison.png", bbox_inches="tight")
plt.show()
print("저장: storage_comparison.png")

---

## 3. 처리량(Throughput) 벤치마크

**전체 SigLIP 처리** vs **Gaze 선택 후 SigLIP 처리** 속도를 비교합니다.

In [ ]:
# ── AutoGaze 단독 처리 시간 측정 ─────────────────────
N_WARMUP = 3
N_REPEAT = 10

def measure_latency(fn, n_warmup=N_WARMUP, n_repeat=N_REPEAT) -> Tuple[float, float]:
    """평균 / 표준편차 latency (ms)"""
    for _ in range(n_warmup):
        fn()
    times = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        fn()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))


# AutoGaze (ratio 0.25)
def run_autogaze(ratio=0.25):
    with torch.inference_mode():
        return model({"video": video_batch}, gazing_ratio=ratio, task_loss_requirement=None)

latency_ag_mean, latency_ag_std = measure_latency(lambda: run_autogaze(0.25))
print(f"AutoGaze (ratio=0.25)  : {latency_ag_mean:.1f} ± {latency_ag_std:.1f} ms  "
      f"→ {1000/latency_ag_mean:.1f} videos/sec")

In [ ]:
# ── SigLIP 처리 시간 측정 (선택적 — SigLIP 없으면 추정치 사용) ─
try:
    from transformers import AutoImageProcessor
    from autogaze.vision_encoders.siglip import SiglipVisionModel

    siglip_transform = AutoImageProcessor.from_pretrained("google/siglip2-base-patch16-224")
    siglip_model = SiglipVisionModel.from_pretrained(
        "google/siglip2-base-patch16-224",
        scales=model.config.scales,
        attn_implementation="sdpa",
    ).to(device).eval()

    video_siglip = transform_video_for_pytorch(raw_video, siglip_transform)[None].to(device)

    # 전체 패치 인코딩
    def run_siglip_full():
        with torch.inference_mode():
            return siglip_model(video_siglip)

    # Gaze 패치만 인코딩
    gaze_out_25 = run_autogaze(0.25)
    def run_siglip_gaze():
        with torch.inference_mode():
            return siglip_model(video_siglip, gazing_info=gaze_out_25)

    lat_siglip_full_mean, lat_siglip_full_std = measure_latency(run_siglip_full)
    lat_siglip_gaze_mean, lat_siglip_gaze_std = measure_latency(run_siglip_gaze)

    SIGLIP_MEASURED = True
    print(f"SigLIP 전체 패치      : {lat_siglip_full_mean:.1f} ± {lat_siglip_full_std:.1f} ms")
    print(f"SigLIP Gaze 패치 (0.25): {lat_siglip_gaze_mean:.1f} ± {lat_siglip_gaze_std:.1f} ms")
    print(f"SigLIP 속도 향상       : {lat_siglip_full_mean/lat_siglip_gaze_mean:.2f}×")

except Exception as e:
    print(f"SigLIP 로드 실패 ({e.__class__.__name__}: {e})")
    print("→ SigLIP 처리 시간을 이론적으로 추정합니다 (토큰 수에 선형 비례 가정)")
    SIGLIP_MEASURED = False

    # ViT는 attention이 O(n²)이지만 배치/구현에 따라 선형에 가까운 경우가 많음
    # 여기서는 보수적으로 선형 가정
    SIGLIP_FULL_REF_MS = 80.0   # 참고 하드웨어(A100)에서 전체 처리 기준값
    lat_siglip_full_mean = SIGLIP_FULL_REF_MS
    lat_siglip_full_std  = 0.0
    lat_siglip_gaze_mean = {}
    lat_siglip_gaze_std  = {}
    for ratio, res in gaze_results.items():
        scale = res['actual_ratio']
        lat_siglip_gaze_mean[ratio] = SIGLIP_FULL_REF_MS * scale
        lat_siglip_gaze_std[ratio]  = 0.0
    print(f"(추정) SigLIP 전체 패치: {lat_siglip_full_mean:.0f} ms")
    for r, ms in lat_siglip_gaze_mean.items():
        print(f"(추정) SigLIP Gaze {r:.2f}: {ms:.1f} ms  (절약 {(1-gaze_results[r]['actual_ratio'])*100:.0f}%)")

In [ ]:
# ── 파이프라인 전체 Throughput 계산 ──────────────────
print("\n파이프라인별 처리량 (videos/sec, 단일 비디오 기준)")
print("=" * 60)

# 공통: 디코딩 오버헤드 (av.open + read_video_pyav) 측정
container_ref = av.open(str(VIDEO_PATH))
container_ref.close()

def decode_video():
    c = av.open(str(VIDEO_PATH))
    frames = read_video_pyav(c, list(range(min(16, total_frames))))
    c.close()
    return frames

lat_decode_mean, lat_decode_std = measure_latency(decode_video)
print(f"비디오 디코딩 (16프레임): {lat_decode_mean:.1f} ± {lat_decode_std:.1f} ms")

# 시나리오별 total latency
for ratio in GAZING_RATIOS:
    lat_ag = gaze_results[ratio]['latency_ms']

    if SIGLIP_MEASURED:
        lat_siglip_g = lat_siglip_gaze_mean
    else:
        lat_siglip_g = lat_siglip_gaze_mean[ratio]

    # 전략 A: AutoGaze + Gaze SigLIP
    total_gaze = lat_decode_mean + lat_ag + lat_siglip_g

    print(f"\n[ratio={ratio:.2f}] 전략 A (AutoGaze + Gaze SigLIP)")
    print(f"  decode={lat_decode_mean:.0f}ms + autogaze={lat_ag:.0f}ms + siglip={lat_siglip_g:.0f}ms = {total_gaze:.0f}ms")
    print(f"  → {1000/total_gaze:.2f} videos/sec  ({gaze_results[ratio]['actual_ratio']*100:.0f}% 토큰 사용)")

# 비교: 전략 B — 전체 SigLIP (AutoGaze 없음)
total_full = lat_decode_mean + lat_siglip_full_mean
print(f"\n[기준선] 전략 B (AutoGaze 없이 전체 SigLIP)")
print(f"  decode={lat_decode_mean:.0f}ms + siglip={lat_siglip_full_mean:.0f}ms = {total_full:.0f}ms")
print(f"  → {1000/total_full:.2f} videos/sec  (100% 토큰 사용)")

---

## 4. Agent 반복 접근 시나리오

같은 비디오를 **N번 반복 처리**해야 할 때 (예: agent가 여러 질문을 순서대로 처리),  
**Gaze 피처를 캐시해두면 언제부터 이득**이 되는지 계산합니다.

```
전략 C (No Cache):     N × (decode + AutoGaze + SigLIP)  비용
전략 D (Gaze Cache):   1 × (decode + AutoGaze + SigLIP)  + N × load_cache
전략 E (Full Cache):   1 × (decode + SigLIP_full)        + N × load_full_cache
```

In [ ]:
# ── 캐시 I/O 속도 측정 ──────────────────────────────
import tempfile

RATIO_FOCUS = 0.25  # 대표 gazing ratio

# 가상의 gaze 피처 텐서 (실제 SigLIP 없어도 I/O 속도 측정 가능)
n_gaze_tokens = gaze_results[RATIO_FOCUS]["n_gaze"]
n_full_tokens = T_actual * NUM_TOKENS_PER_FRAME

feat_gaze = torch.randn(n_gaze_tokens, SIGLIP_DIM, dtype=torch.float16)
feat_full = torch.randn(n_full_tokens, SIGLIP_DIM, dtype=torch.float16)

# ── 저장 → 로드 속도 측정 ─────────────────────────
with tempfile.TemporaryDirectory() as tmpdir:
    gaze_path = os.path.join(tmpdir, "gaze_feat.pt")
    full_path = os.path.join(tmpdir, "full_feat.pt")

    torch.save(feat_gaze, gaze_path)
    torch.save(feat_full, full_path)

    gaze_disk_bytes = os.path.getsize(gaze_path)
    full_disk_bytes = os.path.getsize(full_path)

    def load_gaze(): return torch.load(gaze_path, weights_only=True)
    def load_full(): return torch.load(full_path, weights_only=True)

    lat_load_gaze_mean, _ = measure_latency(load_gaze)
    lat_load_full_mean, _ = measure_latency(load_full)

print(f"캐시 파일 크기  — Gaze: {fmt_bytes(gaze_disk_bytes)},  Full: {fmt_bytes(full_disk_bytes)}")
print(f"캐시 로드 시간  — Gaze: {lat_load_gaze_mean:.2f} ms,   Full: {lat_load_full_mean:.2f} ms")

In [ ]:
# ── N번 반복 접근 비용 곡선 ──────────────────────────
if SIGLIP_MEASURED:
    lat_siglip_g_focus = lat_siglip_gaze_mean
else:
    lat_siglip_g_focus = lat_siglip_gaze_mean[RATIO_FOCUS]

# 1회 초기 처리 비용
cost_init_nocache = lat_decode_mean + gaze_results[RATIO_FOCUS]['latency_ms'] + lat_siglip_g_focus
cost_init_gazecache = cost_init_nocache      # 한 번 처리 후 저장
cost_init_fullcache = lat_decode_mean + lat_siglip_full_mean

N_range = np.arange(1, 101)

# 총 비용 (ms)
cost_C_nocache    = cost_init_nocache * N_range                                      # 전략 C
cost_D_gazecache  = cost_init_gazecache + lat_load_gaze_mean * N_range              # 전략 D
cost_E_fullcache  = cost_init_fullcache + lat_load_full_mean * N_range              # 전략 E

# 교차점 계산 (No Cache vs Gaze Cache)
# C: cost_init * N  vs  D: cost_init + load * N
# cost_init * N_cross = cost_init + load * N_cross
# N_cross * (cost_init - load) = cost_init
if cost_init_nocache > lat_load_gaze_mean:
    N_cross_gaze = cost_init_gazecache / (cost_init_nocache - lat_load_gaze_mean)
else:
    N_cross_gaze = float('inf')

if cost_init_nocache > lat_load_full_mean:
    N_cross_full = cost_init_fullcache / (cost_init_nocache - lat_load_full_mean)
else:
    N_cross_full = float('inf')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(N_range, cost_C_nocache / 1000,   label="전략 C: No Cache (매번 재처리)",       color="#C44E52", lw=2)
ax.plot(N_range, cost_D_gazecache / 1000, label=f"전략 D: Gaze Cache (ratio={RATIO_FOCUS})", color="#55A868", lw=2)
ax.plot(N_range, cost_E_fullcache / 1000, label="전략 E: Full SigLIP Cache",              color="#4C72B0", lw=2, ls="--")

if np.isfinite(N_cross_gaze) and N_cross_gaze < 100:
    ax.axvline(N_cross_gaze, color="#55A868", ls=":", alpha=0.7)
    ax.text(N_cross_gaze + 1, cost_D_gazecache[int(N_cross_gaze)] / 1000 * 1.05,
            f"BEP @ N={N_cross_gaze:.1f}", color="#55A868", fontsize=9)

ax.set_xlabel("반복 접근 횟수 N")
ax.set_ylabel("누적 처리 시간 (초)")
ax.set_title(f"Agent 반복 접근: 캐시 전략별 누적 비용 ({T_actual}프레임 비디오)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("agent_cache_comparison.png", bbox_inches="tight")
plt.show()

print(f"\nGaze Cache BEP (손익분기점): N ≥ {N_cross_gaze:.1f} 회 반복 시 캐싱이 유리")
print(f"Full Cache BEP              : N ≥ {N_cross_full:.1f} 회")

---

## 5. 대규모 데이터셋 스케일 추정

AutoGaze 학습 데이터셋 규모(예: InternVid 25만 클립 수준) 또는
실제 배포 시 비디오 수가 늘어날 때 **총 저장 용량**을 비교합니다.

In [ ]:
# ── 스케일 파라미터 ──────────────────────────────────
DATASET_SIZES = {
    "소규모 실험\n(1K 클립)": 1_000,
    "중규모\n(100K 클립)": 100_000,
    "AutoGaze 데이터\n(650K 클립)": 650_000,
    "대규모\n(5M 클립)": 5_000_000,
}

# 기준: 예시 비디오 기반 실측값 (16프레임 클립)
per_clip = {
    "mp4": b_mp4,
    "raw_uint8": b_raw_uint8,
    "full_siglip_fp16": b_full_siglip_fp16,
    "gaze_siglip_0.25": b_gaze_siglip_fp16[0.25],
    "gaze_siglip_0.10": b_gaze_siglip_fp16[0.10],
}

print(f"{'데이터셋':^25} {'MP4':>10} {'Raw uint8':>12} {'Full SigLIP':>13} {'Gaze 0.25':>12} {'Gaze 0.10':>12}")
print("-" * 90)

for label, n_clips in DATASET_SIZES.items():
    label_clean = label.replace('\n', ' ')
    row = []
    for key in per_clip:
        total = per_clip[key] * n_clips
        row.append(fmt_bytes(total))
    print(f"{label_clean:<25} {row[0]:>10} {row[1]:>12} {row[2]:>13} {row[3]:>12} {row[4]:>12}")

In [ ]:
# ── 스택 바 차트: 데이터셋 규모별 저장 용량 ─────────
fig, ax = plt.subplots(figsize=(12, 5))

labels_ds = [k.replace('\n', '\n') for k in DATASET_SIZES]
n_clips_list = list(DATASET_SIZES.values())

strategies = {
    "MP4 원본": [per_clip['mp4'] * n / 1024**3 for n in n_clips_list],
    "Raw uint8 (디코딩)": [per_clip['raw_uint8'] * n / 1024**3 for n in n_clips_list],
    "Full SigLIP fp16": [per_clip['full_siglip_fp16'] * n / 1024**3 for n in n_clips_list],
    "Gaze SigLIP 0.25": [per_clip['gaze_siglip_0.25'] * n / 1024**3 for n in n_clips_list],
    "Gaze SigLIP 0.10": [per_clip['gaze_siglip_0.10'] * n / 1024**3 for n in n_clips_list],
}

colors_ds = ["#4C72B0", "#DD8452", "#8172B2", "#55A868", "#2E8B57"]
x = np.arange(len(labels_ds))
w = 0.15

for i, (name, vals) in enumerate(strategies.items()):
    offset = (i - 2) * w
    bars = ax.bar(x + offset, vals, w, label=name, color=colors_ds[i], edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(labels_ds, fontsize=9)
ax.set_ylabel("총 저장 용량 (GB)")
ax.set_title("데이터셋 규모별 저장 전략 비교")
ax.set_yscale("log")
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, axis="y", alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_bytes(int(v * 1024**3))))

plt.tight_layout()
plt.savefig("dataset_scale_storage.png", bbox_inches="tight")
plt.show()

---

## 6. Throughput 종합 비교

단순 latency가 아닌 **실제 배포 시나리오별 처리량**을 비교합니다.

In [ ]:
# ── 처리량 비교 표 ──────────────────────────────────
print("\n처리 시나리오별 Throughput 요약")
print("=" * 80)
print(f"{'시나리오':<40} {'처리 시간':>10} {'vids/sec':>10} {'토큰 비율':>10} {'피처 크기':>12}")
print("-" * 80)

# A. 매번 디코딩 + 전체 SigLIP (기준)
t_A = lat_decode_mean + lat_siglip_full_mean
print(f"{'A. 매번 디코딩 + Full SigLIP':<40} {t_A:>9.0f}ms {1000/t_A:>10.2f}  {'100%':>10} {fmt_bytes(b_full_siglip_fp16):>12}")

# B. 매번 디코딩 + AutoGaze + Gaze SigLIP (각 ratio)
for ratio in GAZING_RATIOS:
    if SIGLIP_MEASURED:
        lat_sg = lat_siglip_gaze_mean
    else:
        lat_sg = lat_siglip_gaze_mean[ratio]
    t_B = lat_decode_mean + gaze_results[ratio]['latency_ms'] + lat_sg
    pct = f"{gaze_results[ratio]['actual_ratio']*100:.0f}%"
    feat_b = fmt_bytes(b_gaze_siglip_fp16[ratio])
    print(f"{'B. AutoGaze + Gaze SigLIP r='+ str(ratio):<40} {t_B:>9.0f}ms {1000/t_B:>10.2f}  {pct:>10} {feat_b:>12}")

print("-" * 80)

# C. 캐시 히트 (Gaze 피처 파일 로드)
for ratio in GAZING_RATIOS:
    label = f"C. 캐시 히트 (Gaze fp16, r={ratio})"
    feat_c = fmt_bytes(b_gaze_siglip_fp16[ratio])
    pct = f"{gaze_results[ratio]['actual_ratio']*100:.0f}%"
    print(f"{label:<40} {lat_load_gaze_mean:>9.1f}ms {1000/lat_load_gaze_mean:>10.1f}  {pct:>10} {feat_c:>12}")

# D. 캐시 히트 (Full SigLIP 피처 파일 로드)
print(f"{'D. 캐시 히트 (Full SigLIP fp16)':<40} {lat_load_full_mean:>9.1f}ms {1000/lat_load_full_mean:>10.1f}  {'100%':>10} {fmt_bytes(b_full_siglip_fp16):>12}")

print("=" * 80)
print("\n* 캐시 히트가 가장 빠르지만 초기 처리 1회 필요")
print("* Gaze 캐시는 Full 캐시 대비 크기 절감 + 유사 속도")

---

## 7. 요약 및 실용 가이드라인

### 수치 요약

In [ ]:
print("AutoGaze 효율성 요약")
print("=" * 60)
print(f"비디오 : {T_actual} 프레임 × {C}×{H}×{W}")
print(f"전체 토큰 수: {T_actual * NUM_TOKENS_PER_FRAME} 토큰")
print()
print(f"{'gazing_ratio':>14} {'선택 토큰':>10} {'토큰 절감':>10} {'피처 크기':>12} {'피처 절감':>10}")
print("-" * 60)
for ratio in GAZING_RATIOS:
    n_g = gaze_results[ratio]['n_gaze']
    n_t = gaze_results[ratio]['n_total']
    tok_save = 1 - n_g / n_t
    feat_save = 1 - b_gaze_siglip_fp16[ratio] / b_full_siglip_fp16
    print(f"{ratio:>14.2f} {n_g:>10d} {tok_save*100:>9.1f}% {fmt_bytes(b_gaze_siglip_fp16[ratio]):>12} {feat_save*100:>9.1f}%")

print()
print("저장 용량 관점")
print(f"  MP4 원본 대비 SigLIP 전체 피처: {b_full_siglip_fp16/b_mp4:.1f}× 크다")
print(f"  → Gaze 0.25 피처로 줄이면   : {b_gaze_siglip_fp16[0.25]/b_mp4:.2f}× (MP4 대비)")
print(f"  → Gaze 0.10 피처로 줄이면   : {b_gaze_siglip_fp16[0.10]/b_mp4:.2f}× (MP4 대비)")

### 실용 가이드라인

| 시나리오 | 권장 전략 | 이유 |
|---|---|---|
| **1회 처리 후 폐기** | No Cache + AutoGaze | 저장 불필요, 빠른 gaze 처리 |
| **같은 비디오 N≥2 반복 접근** | Gaze 피처 캐시 (fp16) | 디코딩·인코딩 비용 상각, 디스크 절감 |
| **대규모 데이터셋 피처 추출** | Gaze fp16 캐시 (ratio≤0.25) | 스토리지 비용 75%↓, 로딩 속도 향상 |
| **Agent 멀티-쿼리 (비디오 RAG)** | Gaze 피처 캐시 → 메모리 유지 | 쿼리마다 디코딩 없이 KV 재사용 |
| **실시간 스트리밍** | AutoGaze 스트리밍 모드 + 캐시 없음 | KV 캐시로 프레임별 점진적 처리 |

### 저장 전략 결정 기준

```
같은 비디오를 몇 번 접근하나?
    │
    ├── 1회만 → No Cache (AutoGaze + 즉시 처리)
    │
    └── 2회 이상 → 캐시
                │
                ├── 스토리지 제한 있음 → Gaze fp16 캐시 (ratio=0.25)
                │                        전체 대비 ~25% 크기
                │
                └── 스토리지 여유 있음 → Full SigLIP fp16 캐시
                                         (AutoGaze overhead 없이 빠른 로딩)
```

> **핵심 인사이트**:  
> AutoGaze는 단순히 Vision Encoder 연산을 줄이는 것을 넘어,  
> **피처 캐시 크기를 최대 90% 줄여** 디스크 I/O 병목도 해소합니다.  
> 반복 접근이 많은 Agent 워크플로에서 손익분기점(BEP)은 **2~3회**에 불과합니다.